# Generación de datos de entrenamiento

In [1]:
%load_ext autoreload
%autoreload 2

## Generación de instancias

In [ ]:
from generation.instances import generate_instances, FullRandomGenerator

instance_generator = FullRandomGenerator(H=7, S=5, N=25, seed=42)
generate_instances(f"E5-5-RFull", instance_generator, 50000)

In [ ]:
from generation.instances import generate_instances, RandomMovesGenerator

instance_generator = RandomMovesGenerator(H=7, S=5, N=25, r=100, seed=42)
generate_instances(f"E5-5-R100", instance_generator, 50000)

In [ ]:
from generation.instances import generate_instances, RandomMovesGenerator

instance_generator = RandomMovesGenerator(H=7, S=5, N=25, r=50, seed=42)
generate_instances(f"E5-5-R50", instance_generator, 50000)

In [ ]:
from generation.instances import generate_instances, RandomMovesGenerator

instance_generator = RandomMovesGenerator(H=7, S=5, N=25, r=20, seed=42)
generate_instances(f"E5-5-R20", instance_generator, 50000)

In [5]:
from generation.instances import generate_instances, FullRandomGenerator

instance_generator = FullRandomGenerator(H=7, S=5, N=25, seed=43)
generate_instances(f"E5-5-RFull-RL", instance_generator, 1000)

In [ ]:
from generation.instances import generate_instances, RandomMovesGenerator

instance_generator = RandomMovesGenerator(H=7, S=5, N=25, r=100, seed=43)
generate_instances(f"E5-5-R100-RL", instance_generator, 1000)

## Generación de datos (SL)

In [ ]:
from generation.data import generate_data_sl
from generation.adapters.input import EnrichedLayoutAdapter, Layout4D3FAdapter
from generation.adapters.input.stack_features import StackFeatures3FAdapter
from generation.adapters.output import CostAdapter
from solvers.model import ModelSolver

folders = ["E5-5-R20", "E5-5-R50", "E5-5-R100", "E5-5-RFull"]
H = 7
max_steps = 50
input_adapter_config = (EnrichedLayoutAdapter, Layout4D3FAdapter, StackFeatures3FAdapter)
output_adapter_config = (CostAdapter, )
num_workers = 12
output_prefix = "cost"

generate_data_sl(folders, H, max_steps, input_adapter_config, output_adapter_config, num_workers, output_prefix)

Datos guardados en: /home/oscar/Escritorio/CPMP-Framework/data/cost_E5-5-R20.data (Tamaño 49454)
Datos guardados en: /home/oscar/Escritorio/CPMP-Framework/data/cost_E5-5-R50.data (Tamaño 49985)
Datos guardados en: /home/oscar/Escritorio/CPMP-Framework/data/cost_E5-5-R100.data (Tamaño 50000)
Datos guardados en: /home/oscar/Escritorio/CPMP-Framework/data/cost_E5-5-RFull.data (Tamaño 50000)


In [11]:
## Generación de datasets por dificultad
## Costo = número de pasos

from preprocessing.dataset import generate_dataset

data_files = ["cost_E5-5-R20.data", "cost_E5-5-R50.data", "cost_E5-5-R100.data", "cost_E5-5-RFull.data"]
max_size = 20000
cost_intervals = [(1, 10), (11, 20), (21, 40)]

for min_cost, max_cost in cost_intervals:
    output_name = f"cost_E5-5-C{min_cost}-{max_cost}.data"
    generate_dataset(data_files, output_name, min_cost, max_cost, max_size)

Dataset generado exitosamente en: /home/oscar/Escritorio/CPMP-Framework/data/cost_E5-5-C1-10.data (Tamaño 20000)
Dataset generado exitosamente en: /home/oscar/Escritorio/CPMP-Framework/data/cost_E5-5-C11-20.data (Tamaño 20000)
Dataset generado exitosamente en: /home/oscar/Escritorio/CPMP-Framework/data/cost_E5-5-C21-40.data (Tamaño 20000)


In [12]:
# Concatenamos los datasets
from preprocessing.dataset import generate_dataset

data_files = ["cost_E5-5-C1-10.data", "cost_E5-5-C11-20.data", "cost_E5-5-C21-40.data"]
output_name = "cost_E5-5.data"

generate_dataset(data_files, output_name)

Dataset generado exitosamente en: /home/oscar/Escritorio/CPMP-Framework/data/cost_E5-5.data (Tamaño 60000)


## Generación de datos (RL)

In [ ]:
## La generación de datos ocurre dentro del entrenamiento
## NO es necesario generarlos manualmente.

from generation.data import split_instances, generate_data_rl 
from generation.adapters.input import EnrichedLayoutAdapter, Layout4D3FAdapter
from generation.adapters.input.stack_features import StackFeatures3FAdapter
from generation.adapters.output import ActionAdapter
from training.training import load_model
from models.cpmp_transformer_v5 import CPMPTransformer

folders = ["E5-5-test"]
H = 7
max_steps = 50
input_adapter_config = (EnrichedLayoutAdapter, Layout4D3FAdapter, StackFeatures3FAdapter)
output_adapter_config = (ActionAdapter, )
model = load_model(CPMPTransformer, "best_v5")
batch_size = 32
num_workers = 12
output_name = "E5-5-test-rl.data"

# Divide las instancias en dos conjuntos (entrenamiento y validación)
# Asignamos el 80% para entrenamiento y el 20% para validación 
instance_files, _ = split_instances(folders, 0.80, 0.20, 42)

generate_data_rl(instance_files, H, max_steps, input_adapter_config, output_adapter_config, model, batch_size, num_workers, output_name)

In [10]:
## La generación de datos ocurre dentro del entrenamiento
## NO es necesario generarlos manualmente.

from generation.data import split_instances, generate_data_rl 
from generation.adapters.input import EnrichedLayoutAdapter, Layout4D3FAdapter
from generation.adapters.input.stack_features import StackFeatures3FAdapter
from generation.adapters.output import CostAdapter
from training.training import load_model
from models.cpmp_transformer_v5 import CPMPTransformer

folders = ["E5-5-R20", "E5-5-R50", "E5-5-R100", "E5-5-RFull"]

for folder in folders:
    H = 7
    max_steps = 50
    input_adapter_config = (EnrichedLayoutAdapter, Layout4D3FAdapter, StackFeatures3FAdapter)
    output_adapter_config = (CostAdapter, )
    model = load_model(CPMPTransformer, "best_rl")
    batch_size = 32
    num_workers = 12
    output_name = f"cost_{folder}.data"

    instance_files, _ = split_instances([folder], 1, 0, 42)
    instance_files = instance_files[:20000]

    generate_data_rl(instance_files, H, max_steps, input_adapter_config, output_adapter_config, model, batch_size, num_workers, output_name)

Datos guardados en: /home/oscar/Escritorio/CPMP-Framework/data/cost_E5-5-R20.data (Tamaño 19962)
Datos guardados en: /home/oscar/Escritorio/CPMP-Framework/data/cost_E5-5-R50.data (Tamaño 19999)
Datos guardados en: /home/oscar/Escritorio/CPMP-Framework/data/cost_E5-5-R100.data (Tamaño 20000)
Datos guardados en: /home/oscar/Escritorio/CPMP-Framework/data/cost_E5-5-RFull.data (Tamaño 19828)
